# Novel Architecture for Compound-Protein Interaction Prediction

This notebook implements a novel deep learning model for binary compound-protein interaction (CPI) prediction. The architecture features multi-view compound encoding, hierarchical protein encoding, and cross-modal co-attention for biologically motivated interaction modeling.

## Overview

- **Compound Encoder**: Combines graph neural network (GNN) and transformer-based SMILES encoding with cross-attention fusion.
- **Protein Encoder**: Hierarchical encoding using CNN for local motifs and transformer for long-range dependencies, with residual fusion.
- **Cross-Modal Interaction**: Bi-directional co-attention between compound and protein representations.
- **Prediction Head**: Fully connected layers with batch normalization, dropout, and GELU activation, ending with sigmoid for binary classification.

The model is designed to be publication-ready, modular, and extensible to multi-label scenarios.

In [2]:
## 1. Import Libraries

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading and Preprocessing

Load the CSV dataset, preprocess SMILES strings into molecular graphs and tokenized protein sequences, and perform train/validation/test split.

In [7]:
import os

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Load dataset - replace 'dataset.csv' with actual file path
df = pd.read_csv('./dataset/b_cancer/original/data.txt', sep=' ', header=None, names=['smiles', 'protein_sequence', 'label'])

print(df)

# Basic preprocessing
df = df.dropna()
df['label'] = df['label'].astype(int)

print(f"Dataset size: {len(df)}")
print(df.head())

# SMILES preprocessing
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # Get atom features
    atom_features = []
    for atom in mol.GetAtoms():
        features = [
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetHybridization()),
            int(atom.GetIsAromatic()),
            atom.GetTotalNumHs()
        ]
        atom_features.append(features)
    # Get bonds
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])
        bond_type = bond.GetBondTypeAsDouble()
        edge_attr.append([bond_type])
        edge_attr.append([bond_type])
    return {
        'x': torch.tensor(atom_features, dtype=torch.float),
        'edge_index': torch.tensor(edge_index, dtype=torch.long).t(),
        'edge_attr': torch.tensor(edge_attr, dtype=torch.float)
    }

# Protein sequence tokenization
AA_VOCAB = {aa: i for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}
def tokenize_protein(seq):
    return [AA_VOCAB.get(aa, 0) for aa in seq.upper() if aa in AA_VOCAB]

# Apply preprocessing
df['graph'] = df['smiles'].apply(smiles_to_graph)
df['tokens'] = df['protein_sequence'].apply(tokenize_protein)
df = df[df['graph'].notna()]

# Split data
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

                                                 smiles  \
0                         N[C@@H](Cc1ccc(Br)cc1)C(=O)NO   
1      FC(F)(F)C(=O)c1ccncc1NC(=O)c1ccnc(NC(=O)C2CC2)c1   
2     CC(C)(C)OC(=O)Nc1ccc(cc1)-c1cn(CCC[C@H](NC(=O)...   
3     CC[C@@H](CS(=O)(=O)CC1(C)COC1)N1[C@@H]([C@H](C...   
4     COc1ncc(-c2cc3C(=O)N([C@H](c3n2C(C)C)c2ccc(cc2...   
...                                                 ...   
1613  COc1ccccc1-c1nc2C(=O)N(C(c2n1C(C)C)c1ccc(cc1C)...   
1614  COc1ccc(C)nc1-c1nc2nc(nc(N[C@H](C)C3CCC3)c2n1C...   
1615  CO[C@]1(N([C@H](CC(O)=O)c2ccc(Cl)cc2)C(=O)c2cc...   
1616  OC(=O)c1ccc(cc1)N1C[C@H]2[C@H]([C@H](c3cccc(Cl...   
1617  COc1nc(ncc1-c1cc2C(=O)N(C(c2n1C(C)C)c1ccc(cc1)...   

                                       protein_sequence  label  
0     MSAIQAAWPSGTECIAKYNFHGTAEQDLPFCKGDVLTIVAVTKDPN...      1  
1     MAEPRQEFEVMEDHAGTYGLGDRKDQGGYTMHQDQEGDTDAGLKES...      0  
2     MCNTNMSVPTDGAVTTSQIPASEQETLVRPKPLLLKLLKSVGAQKD...      0  
3     MCNTNMSVPTDGAVTTSQIPASEQE

## 3. Compound Encoder: Graph Neural Network

Define a GAT-based GNN to encode molecular graphs from SMILES.

In [8]:
# GAT Layer
class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=8, dropout=0.1):
        super().__init__()
        self.gat = nn.ModuleList([nn.Linear(in_dim, out_dim) for _ in range(heads)])
        self.attn = nn.ModuleList([nn.Linear(2*out_dim, 1) for _ in range(heads)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        h = []
        for i in range(len(self.gat)):
            h_i = self.gat[i](x)
            # Attention computation
            row, col = edge_index
            h_src = h_i[row]
            h_dst = h_i[col]
            attn_input = torch.cat([h_src, h_dst], dim=-1)
            attn = F.leaky_relu(self.attn[i](attn_input))
            attn = F.softmax(attn, dim=0)
            h_agg = torch.zeros_like(h_i)
            h_agg = h_agg.index_add(0, col, attn * h_src)
            h.append(h_agg)
        return torch.cat(h, dim=-1)

# Compound GNN Encoder
class CompoundGNN(nn.Module):
    def __init__(self, in_dim=6, hidden_dim=64, out_dim=128, heads=8):
        super().__init__()
        self.conv1 = GATLayer(in_dim, hidden_dim, heads)
        self.conv2 = GATLayer(hidden_dim*heads, out_dim, heads)
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, graph):
        x, edge_index = graph['x'], graph['edge_index']
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        # Global pooling
        x = x.mean(dim=0, keepdim=True)  # Simple mean pooling
        return x.squeeze(0)

## 4. Compound Encoder: SMILES Transformer

Implement a parallel Transformer encoder for tokenized SMILES strings with positional encoding.

In [9]:
# SMILES vocabulary (simplified)
SMILES_CHARS = 'CNOScnos123456789=#()[]{}+-./\\@'
SMILES_VOCAB = {c: i+1 for i, c in enumerate(SMILES_CHARS)}  # 0 for unknown
SMILES_VOCAB['<pad>'] = 0

def tokenize_smiles(smiles, max_len=100):
    tokens = [SMILES_VOCAB.get(c, 0) for c in smiles[:max_len]]
    tokens += [0] * (max_len - len(tokens))  # pad
    return tokens

# SMILES Transformer Encoder
class SmilesTransformer(nn.Module):
    def __init__(self, vocab_size=len(SMILES_VOCAB), embed_dim=128, num_heads=8, num_layers=4, max_len=100):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = nn.Parameter(torch.randn(1, max_len, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, tokens):
        # tokens: [batch, seq_len]
        x = self.embedding(tokens) + self.pos_encoding[:, :tokens.size(1)]
        x = self.transformer(x)
        return x.mean(dim=1)  # Global average pooling

## 5. Compound Encoder: Fusion with Cross-Attention

Fuse GNN and Transformer embeddings using learnable cross-attention.

In [10]:
# Cross-Attention Fusion
class CrossAttentionFusion(nn.Module):
    def __init__(self, embed_dim=128, num_heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

    def forward(self, gnn_emb, trans_emb):
        # gnn_emb: [embed_dim], trans_emb: [embed_dim]
        # For single sample, unsqueeze
        gnn_emb = gnn_emb.unsqueeze(0).unsqueeze(0)  # [1, 1, embed_dim]
        trans_emb = trans_emb.unsqueeze(0).unsqueeze(0)  # [1, 1, embed_dim]
        attn_out, _ = self.attn(gnn_emb, trans_emb, trans_emb)
        return attn_out.squeeze(0).squeeze(0) + gnn_emb.squeeze(0).squeeze(0)  # Residual fusion

## 6. Protein Encoder: CNN for Local Motifs

Apply CNN layers to tokenized protein sequences for extracting local amino acid motifs.

In [11]:
# Protein CNN Encoder
class ProteinCNN(nn.Module):
    def __init__(self, vocab_size=20, embed_dim=64, kernel_sizes=[3,5,7], num_filters=64, max_len=1000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
        self.pool = nn.AdaptiveMaxPool1d(1)

    def forward(self, tokens):
        # tokens: [seq_len]
        x = self.embedding(tokens).unsqueeze(0).transpose(1,2)  # [1, embed_dim, seq_len]
        conv_outs = [F.relu(conv(x)) for conv in self.convs]
        pooled = [self.pool(out).squeeze(-1).squeeze(0) for out in conv_outs]
        return torch.cat(pooled, dim=-1)  # [num_filters * len(kernel_sizes)]

## 7. Protein Encoder: Transformer for Long-Range Dependencies

Use a Transformer encoder to capture long-range residue interactions in protein sequences.

In [12]:
# Protein Transformer Encoder
class ProteinTransformer(nn.Module):
    def __init__(self, vocab_size=21, embed_dim=128, num_heads=8, num_layers=4, max_len=1000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = nn.Parameter(torch.randn(1, max_len, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, tokens):
        # tokens: [seq_len]
        x = self.embedding(tokens).unsqueeze(0) + self.pos_encoding[:, :tokens.size(0)]
        x = self.transformer(x)
        return x.mean(dim=1).squeeze(0)  # [embed_dim]

## 8. Protein Encoder: Residual Fusion

Combine CNN and Transformer outputs via residual connections.

In [13]:
# Protein Encoder with Residual Fusion
class ProteinEncoder(nn.Module):
    def __init__(self, cnn_out_dim=192, trans_dim=128, out_dim=128):
        super().__init__()
        self.cnn = ProteinCNN()
        self.transformer = ProteinTransformer(embed_dim=trans_dim)
        self.fusion = nn.Linear(cnn_out_dim + trans_dim, out_dim)

    def forward(self, tokens):
        cnn_emb = self.cnn(tokens)
        trans_emb = self.transformer(tokens)
        combined = torch.cat([cnn_emb, trans_emb], dim=-1)
        return self.fusion(combined) + (cnn_emb.mean() + trans_emb) / 2  # Simple residual

## 9. Cross-Modal Interaction Module: Bi-Directional Co-Attention

Implement bi-directional co-attention between compound and protein embeddings.

In [14]:
# Bi-Directional Co-Attention
class BiDirectionalCoAttention(nn.Module):
    def __init__(self, embed_dim=128, num_heads=8):
        super().__init__()
        self.comp_to_prot = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.prot_to_comp = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

    def forward(self, comp_emb, prot_emb):
        comp_emb = comp_emb.unsqueeze(0).unsqueeze(0)
        prot_emb = prot_emb.unsqueeze(0).unsqueeze(0)
        comp_attended, _ = self.comp_to_prot(comp_emb, prot_emb, prot_emb)
        prot_attended, _ = self.prot_to_comp(prot_emb, comp_emb, comp_emb)
        return comp_attended.squeeze(0).squeeze(0), prot_attended.squeeze(0).squeeze(0)

## 10. Interaction Prediction Head

Define fully connected layers with batch normalization, dropout, GELU, and sigmoid for binary prediction.

In [15]:
# Interaction Prediction Head
class PredictionHead(nn.Module):
    def __init__(self, in_dim=256, hidden_dims=[256, 128]):
        super().__init__()
        layers = []
        for i, h in enumerate(hidden_dims):
            layers.append(nn.Linear(in_dim if i == 0 else hidden_dims[i-1], h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(0.3))
        layers.append(nn.Linear(hidden_dims[-1], 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return torch.sigmoid(self.net(x)).squeeze(-1)

## 11. Model Definition

Assemble the complete model by integrating compound encoder, protein encoder, interaction module, and prediction head.

In [16]:
# Complete CPI Model
class CPIModel(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.comp_gnn = CompoundGNN(out_dim=embed_dim)
        self.comp_trans = SmilesTransformer(embed_dim=embed_dim)
        self.comp_fusion = CrossAttentionFusion(embed_dim)
        self.prot_encoder = ProteinEncoder(out_dim=embed_dim)
        self.co_attn = BiDirectionalCoAttention(embed_dim)
        self.head = PredictionHead(in_dim=embed_dim*2)

    def forward(self, graph, smiles_tokens, prot_tokens):
        gnn_emb = self.comp_gnn(graph)
        trans_emb = self.comp_trans(smiles_tokens.unsqueeze(0)).squeeze(0)
        comp_emb = self.comp_fusion(gnn_emb, trans_emb)
        prot_emb = self.prot_encoder(prot_tokens)
        comp_att, prot_att = self.co_attn(comp_emb, prot_emb)
        interaction_emb = torch.cat([comp_att, prot_att], dim=-1)
        return self.head(interaction_emb)

## 12. Training Pipeline

Set up training loop with binary cross-entropy loss, AdamW optimizer, learning rate scheduler, and early stopping.

In [20]:
# df['smiles_tokens'] = df['smiles'].apply(lambda x: tokenize_smiles(x))

# df

In [23]:
for a in train_loader:
    print(a)

KeyError: 'smiles_tokens'

In [24]:
# Update data with SMILES tokens
df['smiles_tokens'] = df['smiles'].apply(lambda x: tokenize_smiles(x))

# CPI Dataset
class CPIDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return row['graph'], torch.tensor(row['smiles_tokens']), torch.tensor(row['tokens']), torch.tensor(row['label'], dtype=torch.float)

# DataLoaders
train_dataset = CPIDataset(train_df)
val_dataset = CPIDataset(val_df)
test_dataset = CPIDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

# Model, Optimizer, etc.
model = CPIModel()
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.BCELoss()

# Early stopping
best_val_loss = float('inf')
patience = 10
counter = 0

# Training loop
for epoch in range(100):
    model.train()
    train_loss = 0
    for graphs, smiles_ts, prot_ts, labels in train_loader:
        optimizer.zero_grad()
        preds = []
        for g, s, p, l in zip(graphs, smiles_ts, prot_ts, labels):
            pred = model(g, s, p)
            preds.append(pred)
        preds = torch.stack(preds)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for graphs, smiles_ts, prot_ts, labels in val_loader:
            preds = []
            for g, s, p, l in zip(graphs, smiles_ts, prot_ts, labels):
                pred = model(g, s, p)
                preds.append(pred)
            preds = torch.stack(preds)
            loss = criterion(preds, labels)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

KeyError: 'smiles_tokens'

## 13. Evaluation Metrics

Compute and report accuracy, ROC-AUC, precision, recall, F1-score, and MCC on test set.

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pth'))

# Evaluation
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for graphs, smiles_ts, prot_ts, labels in test_loader:
        preds = []
        for g, s, p, l in zip(graphs, smiles_ts, prot_ts, labels):
            pred = model(g, s, p)
            preds.append(pred.item())
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(f"Accuracy: {accuracy_score(all_labels, all_preds > 0.5):.4f}")
print(f"ROC-AUC: {roc_auc_score(all_labels, all_preds):.4f}")
print(f"Precision: {precision_score(all_labels, all_preds > 0.5):.4f}")
print(f"Recall: {recall_score(all_labels, all_preds > 0.5):.4f}")
print(f"F1-Score: {f1_score(all_labels, all_preds > 0.5):.4f}")
print(f"MCC: {matthews_corrcoef(all_labels, all_preds > 0.5):.4f}")

## 14. Interpretability: Attention Weights and Visualization

Save co-attention weights and provide code to visualize important atoms in molecules and residues in proteins.

In [ ]:
# Interpretability - Placeholder for attention weights saving
# Modify BiDirectionalCoAttention to return attention weights
# For visualization, example code

import matplotlib.pyplot as plt

# For molecule visualization
def visualize_molecule(smiles, important_atoms):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        from rdkit.Chem.Draw import rdMolDraw2D
        drawer = rdMolDraw2D.MolDraw2DSVG(300, 300)
        drawer.DrawMolecule(mol, highlightAtoms=important_atoms)
        drawer.FinishDrawing()
        svg = drawer.GetDrawingText()
        with open('molecule_highlight.svg', 'w') as f:
            f.write(svg)
        print("Molecule visualization saved to molecule_highlight.svg")

# For protein sequence visualization
def visualize_protein_sequence(sequence, important_residues):
    colors = ['red' if i in important_residues else 'black' for i in range(len(sequence))]
    fig, ax = plt.subplots(figsize=(len(sequence)/10, 2))
    for i, aa in enumerate(sequence):
        ax.text(i, 0, aa, color=colors[i], fontsize=12, ha='center')
    ax.set_xlim(-0.5, len(sequence)-0.5)
    ax.set_ylim(-0.5, 0.5)
    ax.axis('off')
    plt.savefig('protein_highlight.png')
    print("Protein visualization saved to protein_highlight.png")

# Example usage
# important_atoms = [0, 2, 5]  # Indices of important atoms
# visualize_molecule('CCO', important_atoms)
# important_residues = [10, 20, 30]  # Indices of important residues
# visualize_protein_sequence('ACDEFGHIKLMNPQRSTVWY', important_residues)